# Lab 11: Government Data APIs

*INFO 4617/5617 · Week 11 · Chapter 11, [Government Data APIs](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html)*

Today we code together. Each step below says what we're building and why, and its code cell holds only the comments that frame it: we write the code live, in class. Nothing here is turned in. After each step, a **💬 Discuss** box holds questions to talk over before the next one; click it to open it.

We'll answer the chapter's question with two public APIs: **how did Boulder's housing come to look the way it does?** The Census Bureau tells us what the housing stock is today; FRED, the Federal Reserve Bank of St. Louis's data service, tells us about the economy it was built in. Along the way, short ⏱ microsprints ask you to change one thing, alone or in pairs, and share what you find.

**Before class:** open this notebook in the `webdata` environment (chapter 1), and have your email address ready for the User-Agent. Request free keys from the [Census Bureau](https://api.census.gov/data/key_signup.html) and [FRED](https://fred.stlouisfed.org/docs/api/api_key.html) (and, for the optional Step 9, the [FEC](https://api.open.fec.gov/developers/)), and set them as the environment variables `CENSUS_API_KEY` and `FRED_API_KEY` before you start Jupyter; *Missing Manual* Chapter 34 shows how. Approval can take a few minutes, so don't wait until class.

On Friday, run every cell above where we stopped (**Run → Run All Above Selected Cell**).

## Step 0 — Set up, and keep the keys out of the notebook (5 minutes)

An API key identifies you, counts your requests, and is yours alone. A key typed into a notebook travels with the notebook: into a GitHub repository, a class submission, a classmate's laptop. So read the keys from environment variables, as the chapter's ["API Key Management"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html#api-key-management) section does. The chapter checks each key and raises its own error; [`os.environ["NAME"]`](https://docs.python.org/3/library/os.html#os.environ) does the same in one line, since a missing key raises a `KeyError` that names it. (The chapter also shows a second way, a `.gitignore`d `api_keys.json` file; either ends with the same two variables.) Define the User-Agent too: these APIs ask for a key, but every request still says who sent it, in the book's form.

In [ ]:
# Import os, requests, pandas (as pd), and matplotlib.pyplot (as plt)
# Define HEADERS, whose "User-Agent" is "WebDataScience/1.0 (your-email@colorado.edu)", with your email
# Read CENSUS_API_KEY and FRED_API_KEY into census_key and fred_key with os.environ[...]; never print a key
# Your code here

<details>
<summary>💬 Discuss</summary>

- In week 10 we told Wikipedia who we are with a User-Agent. The Census Bureau and FRED also ask for a key: what can a key do that a User-Agent can't?
- Where could a key typed into this notebook end up: a commit, a screenshot, a shared notebook? Which of those could you take back?
- If `FRED_API_KEY` isn't set, what does this cell do? Why is that better than finding out at the first request?

</details>

## Step 1 — The variable catalog (7 minutes)

The [American Community Survey](https://www.census.gov/data/developers/data-sets/acs-5year.html)'s thousands of variables come in groups (the chapter's "Navigating the Variable Catalog"). Group `B25034` is "Year Structure Built", and the group's catalog is itself an API: send it a GET with [`requests.get()`](https://requests.readthedocs.io/en/latest/api/#requests.get) and our `HEADERS`, and read the answer with `.json()`. Print only the estimates, the codes ending in `E`: the chapter skips the margins of error (`M`), but the group also holds annotation variables (`EA`, `MA`) that would triple the list. Browse all the groups at [api.census.gov/data/2022/acs/acs5/groups.html](https://api.census.gov/data/2022/acs/acs5/groups.html).

In [ ]:
# Request "https://api.census.gov/data/2022/acs/acs5/groups/B25034.json" with HEADERS; read the JSON into variables
# Loop over sorted(variables["variables"].items()), and print each code that ends in "E" with its label
# Your code here

<details>
<summary>💬 Discuss</summary>

- What do the parts of `B25034_002E` tell you: the table, the line, and the final letter? Why would an estimate need an annotation variable beside it?
- In week 6, The Numbers hid its table among decoys. How is picking a variable from this catalog like that, and how is it different?
- Every cell today assumes its request works. If the Census Bureau answered a 429, or a web page with status 200, which line would fail, and would week 10's `raise_for_status()` have caught either?

</details>

## Step 2 — Ask about one place (5 minutes)

Places are named by [FIPS codes](https://www.census.gov/library/reference/code-lists/ansi.html): Colorado is state `08`, and Boulder is place `07850`. The chapter's ["Querying for Specific Places"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html#querying-for-specific-places) asks for the place's name and the first few `B25034` estimates, with `get`, `for`, and `in` parameters, and the key. The answer is a list of lists whose first row is the header, and every value is a string, so make a DataFrame and turn the numbers into numbers with [`pd.to_numeric`](https://pandas.pydata.org/docs/reference/api/pandas.to_numeric.html). Keep the chapter's names, `census_url` and `numeric_cols`: we use them again.

In [ ]:
# census_url = "https://api.census.gov/data/2022/acs/acs5"
# params: get="NAME,B25034_001E,B25034_002E,B25034_003E,B25034_004E,B25034_005E", for="place:07850", in="state:08", key=census_key
# Request census_url with params and HEADERS; make df from rows 1 onward, with row 0 as the columns
# numeric_cols: the columns that start with "B25034"; convert them with pd.to_numeric, and show df
# Your code here

<details>
<summary>💬 Discuss</summary>

- In week 5, `urlparse()` took apart a Census URL just like this request's. Which parameters name the variables, which the place, and which must never appear in a printed error?
- In week 7, the Wayback Machine's CDX API sent the same shape: a list of lists with a header row. Why might an API send data this way rather than as nested dictionaries?
- Why do the numbers arrive as strings, and why must a FIPS code like `08` stay one?

</details>

## Step 3 — Compare two places (12 minutes)

Several places can go in one request: `place:07850,45970` is Boulder and Longmont (the chapter's ["Comparing Multiple Locations"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html#comparing-multiple-locations)). Change one parameter and ask again, then [plot](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.plot.bar.html) the two cities' housing by decade built, side by side. The chapter's chart includes the total among the decades, which dwarfs them; leave it out, and label the decades.

In [ ]:
# Set params["for"] to "place:07850,45970" and request again
# Build df_compare as in Step 2, convert numeric_cols, and show it
# Your code here

In [ ]:
# df_plot: df_compare indexed by NAME, the decade columns only (numeric_cols[1:]), transposed; name its rows by decade
# On plt.subplots(), plot df_plot as grouped bars, with a title, axis labels, and a legend titled "City"
# Your code here

### ⏱ Microsprint (6 minutes, alone or in pairs)

A short version of Additional Exercise 1. Cities differ in size, so raw counts don't compare fairly; shares do.

**Do:** ask for the total (`B25034_001E`) and the three decades since 2000 (`_002E` to `_004E`) for five Colorado places, and compute the share of each city's housing built in 2000 or later. The key uses Boulder, Longmont, Fort Collins (`27425`), Lakewood (`43000`), and Castle Rock (`12415`); look up others in the [FIPS code lists](https://www.census.gov/library/reference/code-lists/ansi.html).
**Share:** the city with the largest share, the one with the smallest, and the two shares.

In [ ]:
# Microsprint: set params["get"] to NAME and B25034_001E to B25034_004E, params["for"] to five places, and request
# df_five: indexed by NAME, the four counts converted with pd.to_numeric
# since_2000: 002E + 003E + 004E; share: since_2000 / 001E; show share, largest first
# Your code here

<details>
<summary>💬 Discuss</summary>

- Which of your five cities built the most of its housing since 2000, and which the least? What might explain the difference?
- Longmont out-built Boulder in every decade the chart shows. Is that a fair comparison, and what does the share add?
- These are 2018–2022 five-year estimates. What does that mean for a count labeled "2020 or later"?

</details>

## Step 4 — Sanity-check a number (8 minutes)

A variable with the right-sounding label can still be the wrong one. The chapter's ["Finding the Right Variable"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html#finding-the-right-variable) ends with a practical tip: before you trust a pipeline, retrieve one value you can judge. Ask for Boulder's median household income, [`B19013_001E`](https://api.census.gov/data/2022/acs/acs5/groups/B19013.html). Does it look like a city's median income: tens of thousands of dollars, not 7 or 70 million?

In [ ]:
# params: get="NAME,B19013_001E", for="place:07850", in="state:08", key=census_key; request it
# Print the city and its median household income, with a dollar sign and commas
# Your code here

### ⏱ Microsprint (4 minutes, alone or in pairs)

Check the step's result another way. Every estimate has a [margin of error](https://www.census.gov/content/dam/Census/library/publications/2020/acs/acs_general_handbook_2020_ch07.pdf), the variable with the same code ending in `M`; the American Community Survey gives it at 90% confidence.

**Do:** ask for `B19013_001E` and `B19013_001M` for Boulder and Longmont, and compute each city's interval: the estimate minus and plus its margin.
**Share:** the two intervals, and whether they overlap.

In [ ]:
# Microsprint: set params["get"] to NAME, B19013_001E, and B19013_001M, params["for"] to "place:07850,45970", and request
# incomes: indexed by NAME, the two columns converted; add low (estimate - margin) and high (estimate + margin)
# Show incomes: do the two intervals overlap?
# Your code here

<details>
<summary>💬 Discuss</summary>

- Does Boulder's median income look right? What did you compare it with?
- Did the two intervals overlap? If they had, what could you no longer say about which city's households earn more?
- Which checks like this one could a pipeline run on its own, every time it runs?

</details>

## Step 5 — A FRED series (7 minutes)

[FRED](https://fred.stlouisfed.org/) holds hundreds of thousands of time series, each with an ID you find by searching its website. The chapter's ["Finding Series IDs"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html#finding-series-ids) finds `MHICO08013A052NCEN`, median household income for Boulder County from the Census Bureau's SAIPE program: the ID embeds the state (`CO`) and the county's FIPS code (`08013`). Ask the [`series/observations`](https://fred.stlouisfed.org/docs/api/fred/series_observations.html) endpoint for it, as the chapter does, and fix `observation_end` at a past date so a rerun gives the same answer. FRED writes a missing value as `"."`, so convert with `errors="coerce"`.

In [ ]:
# fred_url = "https://api.stlouisfed.org/fred/series/observations"
# params: series_id="MHICO08013A052NCEN", api_key=fred_key, file_type="json", observation_end="2026-09-30"; request it
# df_income from data["observations"]; convert date with pd.to_datetime and value with pd.to_numeric(errors="coerce"); show the last rows
# Your code here

<details>
<summary>💬 Discuss</summary>

- This series is for Boulder County, `08013`; Step 2 asked about Boulder city, place `07850`. Why do the two incomes differ, and which would you report?
- FRED writes a missing value as `"."`. What would happen without `errors="coerce"`, and what would happen if you dropped it silently?
- FRED republishes the Census Bureau's SAIPE numbers. When would you go to the original source instead?

</details>

## Step 6 — One function for FRED, and let the server do the math (12 minutes)

We'll ask FRED for five more series, and every request repeats Step 5's lines with a different series. So gather those lines into one function, `fred_series()`, written live: what changes from call to call becomes an argument (the series, the start date, `units`, and `frequency`), and the rest stays as Step 5 wrote it. A `frequency` of `None` means "as published": requests [leaves out](https://requests.readthedocs.io/en/latest/user/quickstart/#passing-parameters-in-urls) a parameter whose value is `None`.

Then use it. FRED can transform a series before it sends it (the chapter's ["Server-Side Transformations"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html#server-side-transformations)). With [`units="pc1"`](https://fred.stlouisfed.org/docs/api/fred/series_observations.html#units), the percent change from a year before, the Consumer Price Index (`CPIAUCSL`) comes back as the inflation rate. Plot it since 2020, with a line at the Federal Reserve's 2% target.

In [ ]:
# Define fred_series(series_id, start, units="lin", frequency=None): Step 5's lines, gathered
#   params as in Step 5, plus observation_start=start, units=units, and frequency=frequency
#   Request it, build the DataFrame and convert as in Step 5, and return its date and value columns
# Your code here

In [ ]:
# Ask fred_series() for CPIAUCSL since 2020-01-01, with units="pc1", into df_inflation
# On plt.subplots(), plot value against date, with a dashed red line at 2 labeled "2% target"; title, axis labels, legend
# Your code here

### ⏱ Microsprint (4 minutes, alone or in pairs)

Check the server's math.

**Do:** find the month inflation peaked with [`idxmax()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.idxmax.html), ask for `CPIAUCSL`'s index levels (no `units`), and compute that month's percent change from a year before yourself.
**Share:** the month, FRED's number, and yours.

In [ ]:
# Microsprint: peak, the row of df_inflation with the largest value (idxmax)
# levels: fred_series() for CPIAUCSL since 2021-01-01 (no units), as a Series indexed by date
# Compute (level that month / level a year before - 1) * 100, and print it beside FRED's value
# Your code here

<details>
<summary>💬 Discuss</summary>

- Did your number match FRED's? What did `units="pc1"` save you, and what do you give up by letting the server do the math?
- Starting the chart in 2020 shapes what it shows. How would it look from 2015, or from 2022?
- In week 4 we peeled JSON one level at a time. Step 5's observations held more than a date and a value: what else, and why might FRED record it?

</details>

## Step 7 — Denver against the nation (11 minutes)

National numbers hide local ones (the chapter's "Comparing CPI Across Locations"). The chapter asks for Denver's CPI as `CUURS49BSA0`, but FRED's CPI IDs reuse the Bureau of Labor Statistics' area codes, and in the [BLS's list](https://download.bls.gov/pub/time.series/cu/cu.area) `S49B` is San Francisco–Oakland–Hayward; Denver–Aurora–Lakewood is `S48B`. So ask FRED's [`series`](https://fred.stlouisfed.org/docs/api/fred/series.html) endpoint, which describes a series rather than sending its data, what the chapter's ID is. The Denver-area CPI we'll use is `CUUSA433SA0`, which the Bureau of Labor Statistics publishes once a year. It isn't seasonally adjusted, so compare it, as the chapter says, with the national series that isn't either, `CPIAUCNS`, which FRED can average into years for you ([`frequency="a"`](https://fred.stlouisfed.org/docs/api/fred/series_observations.html#frequency)). Ask for both as year-over-year changes since 2000, [merge](https://pandas.pydata.org/docs/reference/api/pandas.merge.html) them on date, and plot them together. The merge keeps only the years both series have.

In [ ]:
# Ask FRED's series endpoint, "https://api.stlouisfed.org/fred/series", about the chapter's ID, "CUURS49BSA0"
# Send series_id, api_key=fred_key, and file_type="json" with HEADERS; print the JSON: which city is it?
# Your code here

In [ ]:
# df_denver: CUUSA433SA0; df_national: CPIAUCNS with frequency="a"; both since 2000-01-01, with units="pc1"
# df_cpi_compare: merge them on date, with suffixes ("_denver", "_national"); show its last rows
# Your code here

In [ ]:
# On plt.subplots(), plot both lines and a dashed gray line at 2; title, axis labels, legend
# Your code here

### ⏱ Microsprint (5 minutes, alone or in pairs)

A short version of Additional Exercise 2, outside Colorado.

**Do:** search [FRED](https://fred.stlouisfed.org/) for "consumer price index all items" and another metro area, and pick its annual series (the key uses the Seattle area, `CUUSA423SA0`). Merge it with `df_national`, and find the year it ran farthest above the nation.
**Share:** the metro area, its series ID, the year, and the gap in percentage points.

In [ ]:
# Microsprint: ask fred_series() for your metro area's annual CPI since 2000-01-01, with units="pc1"
# Merge it with df_national on date, and compute metro minus national for each year
# Print the year with the largest gap, and the gap
# Your code here

<details>
<summary>💬 Discuss</summary>

- Why compare a series that isn't seasonally adjusted with another that isn't? Denver's comes once a year, so we averaged the nation's into years: what did averaging hide?
- Which metro areas did the class find running hottest, and in which years? Is there a pattern?
- The chapter's Denver ID names another city, and its code would still have run and drawn a chart. What would have caught it, and what does that say about trusting code you copy, even from a textbook?

</details>

## Step 8 — Put the two together (11 minutes)

The Census says what Boulder's housing is; FRED says what borrowing cost when it was built (the chapter's ["Combining Census and FRED Data"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html#combining-census-and-fred-data)). Ask the Census for Boulder's housing by decade back to the 1970s (`B25034_002E` to `B25034_007E`), and FRED for the 30-year mortgage rate as yearly averages (`MORTGAGE30US`, `frequency="a"`) since 1970. Plot them [side by side](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.subplots.html).

One trap: the Census marks a value it can't publish with a code such as `-666666666`, not a blank. The chapter's `census_int()` turns such codes into `None` one value at a time; pandas's [`mask()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.mask.html) does it for a whole Series in one line, since no count is below zero.

In [ ]:
# housing_params: NAME and B25034_002E through B25034_007E for Boulder (place:07850 in state:08); request it into housing_data
# counts: pd.to_numeric of housing_data[1][1:7], as a Series indexed by decade: 2020, 2010, 2000, 1990, 1980, 1970
# Mask every count below zero, and show counts
# Your code here

In [ ]:
# df_mortgage: fred_series() for MORTGAGE30US since 1970-01-01, with frequency="a"
# Two panels from plt.subplots(1, 2): bars of counts by decade, and the mortgage rate over time; titles and axis labels on both
# Your code here

### ⏱ Microsprint (6 minutes, alone or in pairs)

A short version of Additional Exercise 4: put the two sources in one table.

**Do:** average the mortgage rate within each decade from the 1970s to the 2010s, put it beside the units Boulder built that decade, and compute the [correlation](https://pandas.pydata.org/docs/reference/api/pandas.Series.corr.html).
**Share:** the correlation, its sign, and how many data points it rests on.

In [ ]:
# Microsprint: add a decade column to df_mortgage (the year // 10 * 10), and average value by decade
# df_decades: those averages beside counts in one DataFrame, from 1970 to 2010
# Print the correlation between its two columns and the number of decades; show df_decades
# Your code here

<details>
<summary>💬 Discuss</summary>

- The chapter says low-rate decades built more. Does Boulder's table agree, and what can a correlation from five points tell you?
- The Census counts only the homes still standing in 2022. How does that bias a comparison of the 1970s with the 2010s?
- In week 9 we checked PDF extractions with counts and sums. What check would have caught a `-666666666` before it reached a chart?

</details>

## Step 9 — Optional: campaign finance from the FEC (if there's time)

The Federal Election Commission's [API](https://api.open.fec.gov/developers/) needs a third key, `FEC_API_KEY` (the chapter's ["FEC API"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html#fec-api)). Search for candidates by name, then ask for one candidate's totals. Candidate IDs start with the office sought (`P` for president, `S` for Senate, `H` for House); committee IDs start with `C`. Contribution records name individual donors: count them, don't print them.

In [ ]:
# Read FEC_API_KEY into fec_key with os.environ[...]
# Request https://api.open.fec.gov/v1/candidates/search/ with q="Biden", office="P", election_year=2024, per_page=5, api_key=fec_key
# Print each result's name, party_full, and candidate_id
# Your code here

In [ ]:
# Request https://api.open.fec.gov/v1/candidate/P80000722/totals/ with api_key=fec_key and election_year=2024
# Print each result's cycle, receipts, and disbursements, with dollar signs and commas
# Your code here

<details>
<summary>💬 Discuss</summary>

- Campaign finance records name donors. How does that change what you can publish, compared with counts of homes?
- The FEC's data exists because a 1971 law requires it. In week 3's terms, which value does that serve, and what pressure could erode it?

</details>

## Step 10 — What did we learn? (5 minutes)

In a few sentences, as comments:

- Which decade built the most of Boulder's housing, and which the least? Does the mortgage rate explain the pattern, or does something else, such as the city's limits on growth?
- When did Denver's inflation run above the nation's, and by how much at most?
- Where did each API make you look something up: a variable code, a series ID, a FIPS code? Which lookup went wrong, and how did you find out?

**Recommended Exercises ahead** (in the chapter's companion notebook, due Sunday):

- Step 1, load your keys and confirm them without printing them: Step 0 read them from the environment, and a missing one stops the cell.
- Step 2, query every Colorado county: Steps 2 and 3 built the `get`, `for`, and `in` parameters, and Step 1 the catalog for choosing a variable.
- Step 3, build and clean the DataFrame: Step 2's header row and `pd.to_numeric`, and Step 8's `mask()` for codes like `-666666666`.
- Step 4, a bar chart of the top 20 counties: Step 3's labeled bar chart.
- Step 5, a related FRED series for Colorado: Step 5's request and Step 6's `fred_series()`, and Step 7's microsprint, which searched FRED for a series ID.
- Step 6, interpret: Step 4's sanity check and margins of error, and Step 8's question of what two sources side by side can and can't show.

In [ ]:
# Your answers here, as comments


## Friday: from the lab to the textbook (15 minutes)

Chapter 11 is the textbook's, and you can improve it. This lab turned up several candidates; pick one, or something you found yourself, and propose a revision as [`handouts/common/revision-framework.md`](../../handouts/common/revision-framework.md) describes.

- **A series ID for the wrong city.** The chapter's Denver series, `CUURS49BSA0`, is San Francisco's in the Bureau of Labor Statistics' area codes (Step 7). What should the chapter use instead, and what should it say about the series' frequency?
- **A paragraph the data doesn't support.** After its two-panel chart, the chapter says Boulder saw "significant construction in the 1970s and 1990s," and that low-rate decades built more. Compare that with Step 8's chart and its microsprint.
- **A 200 that isn't data.** The Census Bureau redirects a bad key to a web page, and FRED answers during maintenance with one; both arrive with status 200. The chapter's code, like ours, calls `.json()` straight away, and its "Common Issues to Debug" doesn't mention this. What would a reader see, and what should the chapter say?
- **A microsprint that would make a good exercise.** The margin-of-error check (Step 4) or the hand calculation of `pc1` (Step 6).